# E6 - Validacion y refinamiento del pairing axial Al-Kafri/Sudirman

Este notebook valida el pairing preliminar generado en el notebook 13. No entrena modelos y no asume que `pairing_v1` sea correcto. El objetivo es decidir si existe una regla estructural suficientemente confiable para crear un `pairing_refined_v1`, o documentar la ambiguedad y recomendar revision/manual restructuring.

In [ ]:
try:
    import pydicom  # noqa: F401
    import skimage  # noqa: F401
except Exception:
    %pip install -q pydicom scikit-image

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from scipy.io import loadmat
from skimage.measure import label
from skimage.transform import resize
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 180)

## Montaje de Drive y rutas

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No se monto Google Drive automaticamente. Si estas en Colab, montalo manualmente.")
    print("Detalle:", repr(exc))

In [ ]:
ALKAFRI_ROOT = Path("/content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI")
EXTRACTED_ROOT = ALKAFRI_ROOT / "extracted"
NESTED_EXTRACTED_ROOT = EXTRACTED_ROOT / "_nested"
INVENTORY_RESULTS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_inventory")
PAIRING_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing")
PAIRING_V1_ROOT = ALKAFRI_ROOT / "processed" / "pairing_v1"
PAIRING_VALIDATION_ROOT = Path("/content/drive/MyDrive/PFI_MVP/results/E6_alkafri_pairing_validation")
PAIRING_REFINED_ROOT = ALKAFRI_ROOT / "processed" / "pairing_refined_v1"
FIGURES_ROOT = Path("/content/drive/MyDrive/PFI_MVP/figures")
DOCS_ROOT = Path("/content/drive/MyDrive/PFI_MVP/docs")

for path in [PAIRING_VALIDATION_ROOT, PAIRING_REFINED_ROOT, FIGURES_ROOT, DOCS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "extracted_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_extracted_file_inventory.csv",
    "series_orientation": INVENTORY_RESULTS_ROOT / "E6_alkafri_series_orientation_candidates.csv",
    "ground_truth_inventory": INVENTORY_RESULTS_ROOT / "E6_alkafri_ground_truth_inventory.csv",
    "inventory_report": INVENTORY_RESULTS_ROOT / "E6_alkafri_inventory_report.json",
    "axial_ima_candidates": PAIRING_ROOT / "E6_alkafri_axial_ima_candidates.csv",
    "axial_series_summary": PAIRING_ROOT / "E6_alkafri_axial_series_summary.csv",
    "ground_truth_real_files": PAIRING_ROOT / "E6_alkafri_ground_truth_real_files.csv",
    "ground_truth_path_tokens": PAIRING_ROOT / "E6_alkafri_ground_truth_path_tokens.csv",
    "image_path_tokens": PAIRING_ROOT / "E6_alkafri_image_path_tokens.csv",
    "png_ground_truth_summary": PAIRING_ROOT / "E6_alkafri_png_ground_truth_summary.csv",
    "strategy_a": PAIRING_ROOT / "E6_alkafri_pairing_strategy_a_patient.csv",
    "strategy_b": PAIRING_ROOT / "E6_alkafri_pairing_strategy_b_tokens.csv",
    "strategy_c": PAIRING_ROOT / "E6_alkafri_pairing_strategy_c_shape.csv",
    "strategy_d": PAIRING_ROOT / "E6_alkafri_pairing_strategy_d_source_metadata.csv",
    "pairing_candidates": PAIRING_ROOT / "E6_alkafri_pairing_candidates.csv",
    "pairing_sanity": PAIRING_ROOT / "E6_alkafri_pairing_sanity_checks.json",
    "pairing_report": PAIRING_ROOT / "E6_alkafri_pairing_report.json",
}
PAIRING_V1_INDEX = PAIRING_V1_ROOT / "E6_alkafri_processed_pairing_v1_index.csv"

missing = {k: str(v) for k, v in INPUTS.items() if not v.exists()}
if missing:
    raise FileNotFoundError(missing)
print("PAIRING_VALIDATION_ROOT:", PAIRING_VALIDATION_ROOT)

## Carga de outputs de notebooks 12 y 13

In [ ]:
def read_csv_safe(path):
    return pd.read_csv(path) if Path(path).exists() else pd.DataFrame()


extracted_inventory_df = read_csv_safe(INPUTS["extracted_inventory"])
series_orientation_df = read_csv_safe(INPUTS["series_orientation"])
ground_truth_inventory_df = read_csv_safe(INPUTS["ground_truth_inventory"])
axial_ima_df = read_csv_safe(INPUTS["axial_ima_candidates"])
axial_series_summary_df = read_csv_safe(INPUTS["axial_series_summary"])
gt_real_df = read_csv_safe(INPUTS["ground_truth_real_files"])
gt_path_tokens_df = read_csv_safe(INPUTS["ground_truth_path_tokens"])
image_path_tokens_df = read_csv_safe(INPUTS["image_path_tokens"])
png_ground_truth_summary_df = read_csv_safe(INPUTS["png_ground_truth_summary"])
strategy_a_df = read_csv_safe(INPUTS["strategy_a"])
strategy_b_df = read_csv_safe(INPUTS["strategy_b"])
strategy_c_df = read_csv_safe(INPUTS["strategy_c"])
strategy_d_df = read_csv_safe(INPUTS["strategy_d"])
pairing_candidates_df = read_csv_safe(INPUTS["pairing_candidates"])
pairing_v1_index_df = read_csv_safe(PAIRING_V1_INDEX)

with open(INPUTS["inventory_report"], "r", encoding="utf-8") as f:
    inventory_report = json.load(f)
with open(INPUTS["pairing_sanity"], "r", encoding="utf-8") as f:
    pairing_sanity = json.load(f)
with open(INPUTS["pairing_report"], "r", encoding="utf-8") as f:
    pairing_report = json.load(f)

summary = {
    "axial_ima_candidates": len(axial_ima_df),
    "axial_series": len(axial_series_summary_df),
    "ground_truth_real": len(gt_real_df),
    "final_ground_truth_png": int(gt_real_df.get("relative_path", pd.Series(dtype=str)).astype(str).str.contains("05_Final_Ground_Truth_Data|final", case=False, na=False).sum()) if len(gt_real_df) else 0,
    "manual_label_png": int(gt_real_df.get("relative_path", pd.Series(dtype=str)).astype(str).str.contains("03_Manual_Label_Data|manual", case=False, na=False).sum()) if len(gt_real_df) else 0,
    "pairing_candidates": len(pairing_candidates_df),
    "pairing_v1_samples": len(pairing_v1_index_df),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Diagnostico del pairing actual

In [ ]:
diagnosis_rows = []
def add_diag(metric, value):
    diagnosis_rows.append({"metric": metric, "value": value})

add_diag("total_pairing_candidates", len(pairing_candidates_df))
for col in ["confidence", "pairing_strategy", "needs_manual_review", "image_shape", "mask_shape", "shape_match"]:
    if col in pairing_candidates_df.columns:
        counts = pairing_candidates_df[col].value_counts(dropna=False).head(50)
        for key, value in counts.items():
            add_diag(f"{col}={key}", int(value))

pairing_current_diagnosis_df = pd.DataFrame(diagnosis_rows)
pairing_current_diagnosis_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_current_diagnosis.csv"
pairing_current_diagnosis_df.to_csv(pairing_current_diagnosis_csv_path, index=False)

current_summary = {
    "total": int(len(pairing_candidates_df)),
    "confidence_counts": pairing_candidates_df["confidence"].value_counts(dropna=False).to_dict() if "confidence" in pairing_candidates_df else {},
    "strategy_counts": pairing_candidates_df["pairing_strategy"].value_counts(dropna=False).head(30).to_dict() if "pairing_strategy" in pairing_candidates_df else {},
    "needs_manual_review": pairing_candidates_df["needs_manual_review"].value_counts(dropna=False).to_dict() if "needs_manual_review" in pairing_candidates_df else {},
    "examples": {
        conf: pairing_candidates_df[pairing_candidates_df.get("confidence", pd.Series(dtype=str)).eq(conf)].head(5).to_dict(orient="records")
        for conf in ["baja", "media", "alta"]
    },
}
pairing_current_summary_json_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_current_summary.json"
with open(pairing_current_summary_json_path, "w", encoding="utf-8") as f:
    json.dump(current_summary, f, indent=2, ensure_ascii=False)
display(pairing_current_diagnosis_df.head(20))

## Analisis de pairing_v1 si existe

In [ ]:
def normalize_percentile(arr):
    arr = arr.astype(np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    return np.zeros_like(arr, dtype=np.float32) if np.isclose(lo, hi) else np.clip((arr - lo) / (hi - lo), 0, 1)


quality_rows = []
pairing_v1_figures = []
if len(pairing_v1_index_df) > 0:
    sample_df = pairing_v1_index_df.head(20).copy()
    for idx, row in enumerate(sample_df.itertuples(index=False), start=1):
        image_path = getattr(row, "image_npy", None)
        mask_path = getattr(row, "mask_npy", None)
        item = row._asdict()
        item.update({"image_npy_exists": bool(image_path and Path(image_path).exists()), "mask_npy_exists": bool(mask_path and Path(mask_path).exists())})
        try:
            image = np.load(image_path)
            mask = np.load(mask_path)
            fg = mask != 0
            item.update({
                "image_shape_loaded": str(tuple(image.shape)),
                "mask_shape_loaded": str(tuple(mask.shape)),
                "mask_unique_values": json.dumps(np.unique(mask)[:30].tolist()),
                "foreground_ratio": float(np.mean(fg)),
                "mask_empty": bool(np.mean(fg) == 0),
                "mask_full": bool(np.mean(fg) > 0.95),
                "image_valid": bool(np.isfinite(image).all() and np.std(image) > 0),
                "error": None,
            })
            fig_path = FIGURES_ROOT / f"E6_alkafri_pairing_v1_quality_example_{idx:02d}.png"
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(normalize_percentile(image), cmap="gray")
            axes[1].imshow(mask, cmap="viridis")
            axes[2].imshow(normalize_percentile(image), cmap="gray")
            axes[2].imshow(np.ma.masked_where(mask == 0, mask), cmap="autumn", alpha=0.45)
            for ax in axes:
                ax.axis("off")
            fig.suptitle(f"pairing_v1 sample {idx}")
            fig.tight_layout()
            fig.savefig(fig_path, dpi=160, bbox_inches="tight")
            plt.close(fig)
            pairing_v1_figures.append(str(fig_path))
        except Exception as exc:
            item["error"] = repr(exc)
        quality_rows.append(item)

pairing_v1_quality_df = pd.DataFrame(quality_rows)
pairing_v1_quality_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_v1_quality_checks.csv"
pairing_v1_quality_df.to_csv(pairing_v1_quality_csv_path, index=False)

pairing_v1_strategy_counts = pairing_v1_index_df.get("pairing_strategy", pd.Series(dtype=str)).value_counts().to_dict() if len(pairing_v1_index_df) else {}
pairing_v1_reliable = bool(
    len(pairing_v1_quality_df) > 0
    and not all(str(k) == "shape|tokens" for k in pairing_v1_strategy_counts.keys())
    and pairing_v1_quality_df.get("foreground_ratio", pd.Series(dtype=float)).between(0.001, 0.6).mean() > 0.5
)
print("pairing_v1 reliable:", pairing_v1_reliable)
display(pairing_v1_quality_df.head())

## Investigacion de archivos fuente y metadata

In [ ]:
metadata_names = ["Slices Numbers.csv", "T1_Subfolders.csv", "T2_Subfolders.csv", "PNG counts.csv"]
source_metadata_files_df = extracted_inventory_df[
    extracted_inventory_df.get("file_name", pd.Series(dtype=str)).astype(str).isin(metadata_names)
].copy()
source_metadata_files_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_source_metadata_files.csv"
source_metadata_files_df.to_csv(source_metadata_files_csv_path, index=False)

preview_rows = []
for _, row in source_metadata_files_df.iterrows():
    try:
        df = pd.read_csv(row["file_path"])
        preview = df.head(20).to_json(orient="records", force_ascii=False)
        columns = json.dumps(df.columns.tolist(), ensure_ascii=False)
    except Exception as exc:
        preview = repr(exc)
        columns = "[]"
    preview_rows.append({"file_path": row["file_path"], "relative_path": row["relative_path"], "columns": columns, "preview": preview})
source_metadata_tables_preview_df = pd.DataFrame(preview_rows)
source_metadata_tables_preview_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_source_metadata_tables_preview.csv"
source_metadata_tables_preview_df.to_csv(source_metadata_tables_preview_csv_path, index=False)

relevant_dirs = ["03_Extract_Best_Slices", "04_Manual_Label_Checking", "05_Ground_Truth_Development", "06_Semantic_Segmentation"]
snippet_keywords = ["dicom", "mask", "label", "ground", "disc", "thecal", "posterior", "Slices Numbers", "T1_Subfolders", "T2_Subfolders", "imwrite", "png"]
source_m_df = extracted_inventory_df[
    extracted_inventory_df.get("extension", pd.Series(dtype=str)).eq(".m")
    & extracted_inventory_df.get("relative_path", pd.Series(dtype=str)).astype(str).str.contains("|".join(relevant_dirs), case=False, regex=True, na=False)
].copy()

snippet_rows = []
for _, row in source_m_df.iterrows():
    try:
        lines = Path(row["file_path"]).read_text(encoding="utf-8", errors="ignore").splitlines()
        hits = []
        for n, line in enumerate(lines, start=1):
            if any(k.lower() in line.lower() for k in snippet_keywords):
                hits.append(f"{n}: {line.strip()}")
        snippet = "\n".join(hits[:40])
    except Exception as exc:
        snippet = repr(exc)
    snippet_rows.append({"file_path": row["file_path"], "relative_path": row["relative_path"], "snippet": snippet})
source_code_relevant_snippets_df = pd.DataFrame(snippet_rows)
source_code_relevant_snippets_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_source_code_relevant_snippets.csv"
source_code_relevant_snippets_df.to_csv(source_code_relevant_snippets_csv_path, index=False)
display(source_metadata_files_df)

## Tokens especificos de estructura

In [ ]:
def extract_case_id_from_mri_path(text):
    text = str(text)
    m = re.search(r"01_MRI_Data[/\\]([0-9]{4})_", text)
    if not m:
        m = re.search(r"[/\\]([0-9]{4})[^/\\]*[/\\]", text)
    return m.group(1) if m else None


def extract_modality(text):
    t = str(text).lower()
    if re.search(r"\bt1\b|t1_", t):
        return "T1"
    if re.search(r"\bt2\b|t2_", t):
        return "T2"
    return None


def extract_disc_id(text):
    m = re.search(r"\bD[1-9]\b", str(text), flags=re.I)
    return m.group(0).upper() if m else None


def extract_slice_id(text):
    nums = re.findall(r"\d+", str(text))
    return nums[-1] if nums else None


def extract_gt_type(text):
    t = str(text).lower()
    if "05_final_ground_truth_data" in t or "final" in t:
        return "final"
    if "04_intermediary_ground_truth_data" in t or "intermediary" in t:
        return "intermediary"
    if "03_manual_label_data" in t or "manual" in t:
        return "manual"
    return "unknown"


image_specific_rows = []
for _, row in axial_ima_df.iterrows():
    text = f"{row.get('relative_path', '')} {row.get('file_path', '')} {row.get('SeriesDescription', '')}"
    image_specific_rows.append({
        "file_path": row.get("file_path"),
        "relative_path": row.get("relative_path"),
        "case_id": extract_case_id_from_mri_path(text),
        "modality": extract_modality(text),
        "series_uid": row.get("SeriesInstanceUID"),
        "series_description": row.get("SeriesDescription"),
        "instance_number": row.get("InstanceNumber"),
        "slice_id_candidate": str(row.get("InstanceNumber")) if pd.notna(row.get("InstanceNumber")) else extract_slice_id(text),
    })
image_specific_tokens_df = pd.DataFrame(image_specific_rows)
image_specific_tokens_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_image_specific_tokens.csv"
image_specific_tokens_df.to_csv(image_specific_tokens_csv_path, index=False)

gt_specific_rows = []
for _, row in gt_real_df.iterrows():
    text = f"{row.get('relative_path', '')} {row.get('file_path', '')} {row.get('file_name', '')}"
    labeller = re.search(r"Labeller\s*([0-9]+)", text, flags=re.I)
    gt_specific_rows.append({
        "file_path": row.get("file_path"),
        "relative_path": row.get("relative_path"),
        "file_name": row.get("file_name"),
        "extension": row.get("extension"),
        "case_id": re.search(r"[_/\\]([0-9]{4})[_/\\]?", text).group(1) if re.search(r"[_/\\]([0-9]{4})[_/\\]?", text) else None,
        "modality": extract_modality(text),
        "disc_id": extract_disc_id(text),
        "slice_id_candidate": extract_slice_id(text),
        "labeller": labeller.group(1) if labeller else None,
        "gt_type": extract_gt_type(text),
    })
gt_specific_tokens_df = pd.DataFrame(gt_specific_rows)
gt_specific_tokens_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_gt_specific_tokens.csv"
gt_specific_tokens_df.to_csv(gt_specific_tokens_csv_path, index=False)

display(image_specific_tokens_df.head())
display(gt_specific_tokens_df.head())

## Ground truth final y series axiales por caso

In [ ]:
final_gt_df = gt_specific_tokens_df[
    gt_specific_tokens_df["extension"].astype(str).str.lower().eq(".png")
    & gt_specific_tokens_df["gt_type"].eq("final")
].copy()
final_gt_tokens_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_final_gt_png_tokens.csv"
final_gt_df.to_csv(final_gt_tokens_csv_path, index=False)

series_by_case_rows = []
if len(image_specific_tokens_df) > 0:
    merged_img = axial_ima_df.merge(image_specific_tokens_df[["file_path", "case_id", "modality"]], on="file_path", how="left")
    merged_img["is_posdisp_or_localizer"] = merged_img.get("relative_path", pd.Series(dtype=str)).astype(str).str.contains("posdisp|localizer|localiser", case=False, regex=True, na=False)
    for keys, group in merged_img.groupby(["case_id", "modality", "SeriesInstanceUID", "SeriesDescription"], dropna=False):
        case_id, modality, series_uid, desc = keys
        priority = bool(str(desc).lower().find("t2_tse_tra_384") >= 0 or str(desc).lower().find("t1_tse_tra") >= 0)
        series_by_case_rows.append({
            "case_id": case_id,
            "modality": modality,
            "series_uid": series_uid,
            "series_description": desc,
            "n_slices": int(len(group)),
            "is_posdisp_or_localizer": bool(group["is_posdisp_or_localizer"].any()),
            "priority_series": priority,
            "example_path": group["file_path"].iloc[0],
        })
axial_series_by_case_df = pd.DataFrame(series_by_case_rows)
axial_series_by_case_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_axial_series_by_case.csv"
axial_series_by_case_df.to_csv(axial_series_by_case_csv_path, index=False)

print("Final GT:", len(final_gt_df), "Series by case:", len(axial_series_by_case_df))

## Estrategia E estructural refinada

In [ ]:
def image_shape_from_file(path):
    try:
        ds = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
        return (int(ds.Rows), int(ds.Columns))
    except Exception:
        return None


def gt_shape_from_png(path):
    try:
        with Image.open(path) as img:
            w, h = img.size
        return (h, w)
    except Exception:
        return None


source_metadata_available = len(source_metadata_tables_preview_df) > 0
strategy_e_rows = []
candidate_images = image_specific_tokens_df.dropna(subset=["case_id", "modality"]).copy()
candidate_gt = final_gt_df.dropna(subset=["case_id", "modality"]).copy()

for _, img in tqdm(candidate_images.iterrows(), total=len(candidate_images), desc="Strategy E structural"):
    matches = candidate_gt[
        candidate_gt["case_id"].astype(str).eq(str(img["case_id"]))
        & candidate_gt["modality"].astype(str).eq(str(img["modality"]))
    ].copy()
    if len(matches) == 0:
        continue
    img_shape = image_shape_from_file(img["file_path"])
    for _, gt in matches.iterrows():
        gt_shape = gt_shape_from_png(gt["file_path"])
        shape_match = bool(img_shape is not None and gt_shape is not None and tuple(img_shape) == tuple(gt_shape))
        disc_or_slice_match = bool(pd.notna(gt.get("disc_id")) and str(gt.get("disc_id")) != "")
        signals = {
            "case": True,
            "modality": True,
            "disc_or_slice": disc_or_slice_match,
            "shape": shape_match,
            "source_metadata": source_metadata_available,
        }
        if all([signals["case"], signals["modality"], signals["disc_or_slice"], signals["shape"], signals["source_metadata"]]):
            confidence = "alta"
        elif signals["case"] and signals["modality"] and signals["shape"]:
            confidence = "media"
        else:
            confidence = "baja"
        strategy_e_rows.append({
            "image_file_path": img["file_path"],
            "gt_file_path": gt["file_path"],
            "case_id": img["case_id"],
            "modality": img["modality"],
            "disc_id": gt.get("disc_id"),
            "slice_id_candidate": gt.get("slice_id_candidate"),
            "series_description": img.get("series_description"),
            "instance_number": img.get("instance_number"),
            "evidence_case_match": signals["case"],
            "evidence_modality_match": signals["modality"],
            "evidence_disc_or_slice_match": signals["disc_or_slice"],
            "evidence_shape_match": signals["shape"],
            "evidence_source_metadata": signals["source_metadata"],
            "confidence": confidence,
            "reason": json.dumps(signals),
            "needs_manual_review": confidence != "alta",
        })

strategy_e_structural_df = pd.DataFrame(strategy_e_rows)
strategy_e_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_strategy_e_structural.csv"
strategy_e_structural_df.to_csv(strategy_e_csv_path, index=False)
display(strategy_e_structural_df.head())

## Validacion visual y quality checks refinados

In [ ]:
def read_ima_pixels(path):
    ds = pydicom.dcmread(str(path), force=True)
    return normalize_percentile(ds.pixel_array), ds


def read_png_mask(path):
    arr = np.asarray(Image.open(path))
    if arr.ndim == 3:
        return np.any(arr[:, :, :3] != 0, axis=2).astype(np.uint8)
    return arr


quality_rows = []
refined_figures = []
visual_df = strategy_e_structural_df[strategy_e_structural_df["confidence"].isin(["alta", "media"])].head(30).copy() if len(strategy_e_structural_df) else pd.DataFrame()
for idx, row in enumerate(visual_df.itertuples(index=False), start=1):
    try:
        image, ds = read_ima_pixels(row.image_file_path)
        mask = read_png_mask(row.gt_file_path)
        resize_needed = tuple(image.shape[:2]) != tuple(mask.shape[:2])
        overlay_mask = resize(mask, image.shape[:2], order=0, preserve_range=True, anti_aliasing=False).astype(mask.dtype) if resize_needed else mask
        fg = overlay_mask != 0
        fig_path = FIGURES_ROOT / f"E6_alkafri_pairing_refined_example_{idx:02d}.png"
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(image, cmap="gray")
        axes[1].imshow(mask, cmap="viridis")
        axes[2].imshow(image, cmap="gray")
        axes[2].imshow(np.ma.masked_where(overlay_mask == 0, overlay_mask), cmap="autumn", alpha=0.45)
        for ax in axes:
            ax.axis("off")
        fig.suptitle(f"{row.case_id} {row.modality} {row.disc_id} Inst {row.instance_number} {row.confidence}")
        fig.tight_layout()
        fig.savefig(fig_path, dpi=160, bbox_inches="tight")
        plt.close(fig)
        refined_figures.append(str(fig_path))
        quality_rows.append({
            "image_file_path": row.image_file_path,
            "gt_file_path": row.gt_file_path,
            "case_id": row.case_id,
            "modality": row.modality,
            "disc_id": row.disc_id,
            "confidence": row.confidence,
            "foreground_ratio": float(np.mean(fg)),
            "n_components": int(label(fg).max()) if np.any(fg) else 0,
            "unique_values": json.dumps(np.unique(mask)[:30].tolist()),
            "mask_empty": bool(np.mean(fg) == 0),
            "mask_full": bool(np.mean(fg) > 0.95),
            "image_min": float(np.min(image)),
            "image_max": float(np.max(image)),
            "image_mean": float(np.mean(image)),
            "image_std": float(np.std(image)),
            "shape_match": not resize_needed,
            "resize_needed": bool(resize_needed),
            "error": None,
        })
    except Exception as exc:
        quality_rows.append({"image_file_path": getattr(row, "image_file_path", None), "gt_file_path": getattr(row, "gt_file_path", None), "error": repr(exc)})

refined_quality_df = pd.DataFrame(quality_rows)
refined_quality_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_refined_quality_checks.csv"
refined_quality_df.to_csv(refined_quality_csv_path, index=False)
display(refined_quality_df.head())

## Crear pairing_refined_v1 solo si corresponde

In [ ]:
trusted_refined_df = strategy_e_structural_df[strategy_e_structural_df["confidence"].isin(["alta", "media"])].copy() if len(strategy_e_structural_df) else pd.DataFrame()
if len(refined_quality_df) > 0 and "foreground_ratio" in refined_quality_df.columns:
    ok_quality_paths = set(refined_quality_df[
        refined_quality_df["foreground_ratio"].between(0.001, 0.6, inclusive="both")
        & refined_quality_df["error"].isna()
    ]["gt_file_path"].astype(str))
    trusted_refined_df = trusted_refined_df[trusted_refined_df["gt_file_path"].astype(str).isin(ok_quality_paths)].copy()

processed_rows = []
pairing_refined_created = False
if len(trusted_refined_df) >= 50:
    PAIRING_REFINED_ROOT.mkdir(parents=True, exist_ok=True)
    for idx, row in enumerate(trusted_refined_df.itertuples(index=False), start=1):
        sample_id = f"refined_{idx:05d}"
        try:
            image, ds = read_ima_pixels(row.image_file_path)
            mask = read_png_mask(row.gt_file_path)
            image_out = PAIRING_REFINED_ROOT / f"{sample_id}_image.npy"
            mask_out = PAIRING_REFINED_ROOT / f"{sample_id}_mask.npy"
            np.save(image_out, image.astype(np.float32))
            np.save(mask_out, mask)
            processed_rows.append({
                "sample_id": sample_id,
                "image_npy": str(image_out),
                "mask_npy": str(mask_out),
                "image_file_path": row.image_file_path,
                "gt_file_path": row.gt_file_path,
                "confidence": row.confidence,
                "case_id": row.case_id,
                "modality": row.modality,
                "disc_id": row.disc_id,
            })
        except Exception as exc:
            processed_rows.append({"sample_id": sample_id, "image_file_path": row.image_file_path, "gt_file_path": row.gt_file_path, "error": repr(exc)})
    pairing_refined_created = any("image_npy" in r for r in processed_rows)
else:
    print("No hay al menos 50 pares estructurales alta/media con sanity razonable; no se crea pairing_refined_v1.")

processed_refined_index_df = pd.DataFrame(processed_rows)
processed_refined_index_csv_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_processed_pairing_refined_v1_index.csv"
processed_refined_index_df.to_csv(processed_refined_index_csv_path, index=False)

## Sanity checks, reporte y conclusion

In [ ]:
conf_counts = strategy_e_structural_df["confidence"].value_counts().to_dict() if len(strategy_e_structural_df) else {}
problems = []
if not pairing_v1_reliable:
    problems.append("pairing_v1 no se considera confiable automaticamente.")
if len(strategy_e_structural_df) == 0:
    problems.append("No se encontraron pares estructurales.")
if not pairing_refined_created:
    problems.append("No se creo pairing_refined_v1 por cantidad/confianza insuficiente.")

sanity = {
    "n_candidates_notebook_13": int(len(pairing_candidates_df)),
    "n_pairing_v1": int(len(pairing_v1_index_df)),
    "pairing_v1_reliable": bool(pairing_v1_reliable),
    "n_axial_series_by_case": int(len(axial_series_by_case_df)),
    "n_final_gt_by_case": int(len(final_gt_df)),
    "structural_confidence_counts": {str(k): int(v) for k, v in conf_counts.items()},
    "n_pairs_visualized": int(len(refined_figures)),
    "pairing_refined_v1_created": bool(pairing_refined_created),
    "n_processed_refined_samples": int(len(processed_refined_index_df)),
    "problems": problems,
}
sanity_json_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_validation_sanity_checks.json"
with open(sanity_json_path, "w", encoding="utf-8") as f:
    json.dump(sanity, f, indent=2, ensure_ascii=False)

if pairing_refined_created:
    recommendation_14 = "baseline axial"
elif len(strategy_e_structural_df) > 0:
    recommendation_14 = "manual review / dataset restructuring"
else:
    recommendation_14 = "usar solo sagital en MVP inicial y dejar axial como modulo futuro"

exports = {
    "pairing_current_diagnosis": str(pairing_current_diagnosis_csv_path),
    "pairing_current_summary": str(pairing_current_summary_json_path),
    "pairing_v1_quality": str(pairing_v1_quality_csv_path),
    "source_metadata_files": str(source_metadata_files_csv_path),
    "source_metadata_tables_preview": str(source_metadata_tables_preview_csv_path),
    "source_code_relevant_snippets": str(source_code_relevant_snippets_csv_path),
    "image_specific_tokens": str(image_specific_tokens_csv_path),
    "gt_specific_tokens": str(gt_specific_tokens_csv_path),
    "final_gt_tokens": str(final_gt_tokens_csv_path),
    "axial_series_by_case": str(axial_series_by_case_csv_path),
    "strategy_e_structural": str(strategy_e_csv_path),
    "refined_quality": str(refined_quality_csv_path),
    "processed_refined_index": str(processed_refined_index_csv_path),
    "sanity_checks": str(sanity_json_path),
    "figures": refined_figures + pairing_v1_figures,
}

report = {
    "pairing_v1_decision": "reliable" if pairing_v1_reliable else "not_reliable",
    "evidence_used": ["current_pairing_diagnosis", "pairing_v1_quality", "source_metadata", "specific_tokens", "visual_sanity"],
    "refined_strategy": "strategy_e_structural",
    "n_final_pairs": int(len(processed_refined_index_df)),
    "exports": exports,
    "recommendation_for_notebook_14": recommendation_14,
}
report_json_path = PAIRING_VALIDATION_ROOT / "E6_alkafri_pairing_validation_report.json"
with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(json.dumps(report, indent=2, ensure_ascii=False)[:4000])

In [ ]:
strategy_table_md = strategy_e_structural_df["confidence"].value_counts().rename_axis("confidence").reset_index(name="n").to_markdown(index=False) if len(strategy_e_structural_df) else "Sin pares estructurales."
metadata_table_md = source_metadata_files_df[["relative_path", "file_name"]].to_markdown(index=False) if len(source_metadata_files_df) else "No se encontraron metadata CSV fuente."

conclusion_text = f'''# Conclusion tecnica - E6 validacion pairing axial Al-Kafri

## Objetivo

Validar y refinar el emparejamiento imagen-mascara axial propuesto en el notebook 13, sin entrenar modelos.

## Por que se necesito 13b

El notebook 13 produjo un `pairing_v1` preliminar, pero sus senales principales fueron `shape` y `tokens`, generando demasiados candidatos y riesgo de falsos positivos. Por eso se realiza una validacion conservadora con estructura, metadata fuente, tokens especificos y visualizacion.

## Limitacion del pairing del notebook 13

Decision sobre `pairing_v1`: {report["pairing_v1_decision"]}.

## Analisis de codigo fuente y metadata

{metadata_table_md}

## Regla de pairing refinada

Se construyo `strategy_e_structural` usando case_id, modalidad, disco/slice, shape y metadata fuente.

{strategy_table_md}

## Visualizaciones

Figuras generadas: {refined_figures + pairing_v1_figures}

## Decision sobre entrenamiento axial

Recomendacion para notebook 14: `{recommendation_14}`.

`pairing_refined_v1` creado: {pairing_refined_created}.

## Limitaciones

- No se entrenan modelos.
- No se usan pares de baja confianza.
- No se considera shape `(320, 320)` como evidencia suficiente.
- La coherencia anatomica de overlays requiere revision manual/profesional.
- Si el pairing sigue ambiguo, el resultado negativo se documenta como evidencia tecnica.

## Siguiente paso

{recommendation_14}
'''

conclusion_path = DOCS_ROOT / "E6_alkafri_pairing_validation_conclusion.md"
with open(conclusion_path, "w", encoding="utf-8") as f:
    f.write(conclusion_text)
print("Conclusion:", conclusion_path)

## Criterio de aceptacion

Este notebook no entrena modelos, carga outputs de notebooks 12 y 13, diagnostica la ambiguedad del pairing previo, revisa metadata/codigo MATLAB, extrae tokens especificos, genera una estrategia estructural, visualiza pares candidatos, decide si puede crear `pairing_refined_v1` y exporta CSV/JSON/Markdown.